# Replay: Deterministic Artifact Rehydration

Paxman's `replay()` rehydrates a stored artifact byte-for-byte. It does **not** re-invoke the planner, executor, or capabilities. The hash is the contract — if two runs produce the same hash, the normalized data is identical.

This notebook demonstrates determinism: the same input + contract always produces the same artifact hash.

In [ ]:
from pydantic import BaseModel

import paxman
import paxman.contract.adapters.pydantic  # noqa: F401
import paxman.contract.adapters.dict_dsl  # noqa: F401


class Product(BaseModel):
    name: str
    price: str


SAMPLE = "Widget A, price $12.99"

run1 = paxman.normalize(SAMPLE, Product)
run2 = paxman.normalize(SAMPLE, Product)

print(f"run1 hash: {run1.replay_hash}")
print(f"run2 hash: {run2.replay_hash}")
print(f"Deterministic: {run1.replay_hash == run2.replay_hash}")
print()
print(f"Data: {run1.normalized_data}")

## Replay the artifact in memory

`paxman.replay()` takes the artifact and the original contract and returns a verified artifact. If the artifact has been tampered with, it raises `ReplayError`.

In [ ]:
replayed = paxman.replay(run1, Product)

print(f"Original hash: {run1.replay_hash}")
print(f"Replayed hash: {replayed.replay_hash}")
print(f"Hash match: {replayed.replay_hash == run1.replay_hash}")
print()
print(f"Normalized data: {replayed.normalized_data}")
print(f"Status: {replayed.status.name}")

## Replay vs Normalize

**Key difference:** `replay()` is pure deserialization. It returns the same artifact shape without touching the normalization pipeline. This means:
- **Low latency:** microseconds vs milliseconds
- **Deterministic:** same artifact → same output, always
- **Offline:** no capabilities, no planner, no adapters needed at replay time

> **Reference:** See `docs/reference/replay-and-determinism.md` for the full model.

## Try it yourself

- Modify the input string and re-run — the hash should change.
- Try replaying with a different contract — `InvalidContractError` should be raised.
- What happens if you modify `run1.normalized_data` before calling `replay()`?